# Tutorial 0 - Preparing input data with auxiliary tools from the PyPSA ecosystem

The data generated in this tutorial can be also gathered manually from various open-access online sources (see the FAQ document in this repository). Therefore, this tutorial is not strictly necessary to carry out the project assignment.

Still, leveraging tools from the PyPSA ecosystem - as we do here - enables a faster, automated data gathering process, which can be adapted on the fly to different assumptions (e.g., different reference years, different technologies).

In [1]:
import pandas as pd
import powerplantmatching as pm

Cartopy not existent.


## 1. Power Plant data from *powerplantmatching*
For additional information, check out [powerplantmatching.readthedocs.io](https://powerplantmatching.readthedocs.io/en/latest/)

In [2]:
# Load the pre-built European power-plant database
powerplants = pm.powerplants(from_url=True)

# Select power plants located (in this example) in the Netherlands
nl_powerplants = powerplants.loc[
    powerplants["Country"] == "Netherlands"
].copy()

# Filter for power plants that retired before a given year
# We stick to 2019, in this example, to align with the demand and weather data
nl_powerplants = nl_powerplants.loc[
    nl_powerplants["DateOut"].isna()
    | (nl_powerplants["DateOut"] > 2019)
].copy()

# Display the results
display(nl_powerplants.head())

print(f"Number of power plants: {len(nl_powerplants)}")
print(f"Total installed capacity: {nl_powerplants['Capacity'].sum():,.0f} MW")

Matched Data,Name,Fueltype,Technology,Set,Country,Capacity,Efficiency,DateIn,DateRetrofit,DateOut,lat,lon,Duration,Volume_Mm3,DamHeight_m,StorageCapacity_MWh,EIC,projectID
id,,,,,,,,,,,,,,,,,,
137,Maasvlakte Uniper,Hard Coal,Steam Turbine,PP,Netherlands,2306.0,NaN,1975.0,NaN,2029.0,51.958734,4.027208,NaN,NaN,NaN,NaN,{49W000000000102B},"{'GEM': {'G100000106536', 'G100000106537', 'G1..."
490,Borssele 1,Nuclear,Steam Turbine,PP,Netherlands,515.0,NaN,1973.0,NaN,2033.0,51.431200,3.717400,NaN,NaN,NaN,NaN,{49W000000000054X},"{'GEM': {'G100000500496'}, 'ENTSOE': {'49W0000..."
558,Sloe,Natural Gas,CCGT,CHP,Netherlands,870.0,0.59,2009.0,NaN,NaN,51.448200,3.692800,NaN,NaN,NaN,NaN,"{49W000000000117Z, 49W0000000000512}","{'GEM': {'G100000400564', 'G100000400565'}, 'E..."
559,Merwedekanaal,Natural Gas,CCGT,CHP,Netherlands,224.0,NaN,1925.0,NaN,2030.0,52.101700,5.079400,NaN,NaN,NaN,NaN,"{49W000000000090T, 49W000000000091R}","{'GEM': {'G100000400547'}, 'ENTSOE': {'49W0000..."
656,Eemshaven,Hard Coal,Steam Turbine,PP,Netherlands,1600.0,NaN,2014.0,NaN,2029.0,53.435502,6.879276,NaN,NaN,NaN,NaN,{49W000000000066Q},"{'GEM': {'G100000102597', 'G100000102596'}, 'E..."


Number of power plants: 799
Total installed capacity: 46,017 MW


In [3]:
capacity_by_fuel = (
    nl_powerplants
    .groupby("Fueltype")["Capacity"]
    .sum()
    .sort_values(ascending=False)
)

display(capacity_by_fuel)

Fueltype
Natural Gas           17363.50
Wind                  12667.20
Solar                  7366.10
Hard Coal              6007.00
Solid Biomass           717.00
Waste                   696.20
Nuclear                 515.00
Battery                 390.70
Biogas                  160.00
Oil                     107.00
Heat Storage             22.50
Mechanical Storage        3.00
Hydrogen Storage          1.36
Name: Capacity, dtype: float64

In [4]:
capacity_by_technology = (
    nl_powerplants
    .groupby("Technology")["Capacity"]
    .sum()
    .sort_values(ascending=False)
)

display(capacity_by_technology)

Technology
CCGT             15469.0
Steam Turbine     8610.5
PV                7299.7
Offshore          7138.0
Onshore           5505.8
Li                 263.4
Flywheel             3.0
Name: Capacity, dtype: float64

## 2. Downloading techno-economic parameters from Technology Data
You can also see this data online at [PyPSA/technology-data](https://github.com/PyPSA/technology-data/blob/master/outputs/costs_2020.csv)

In [5]:
year = 2020 # Must be one of [2020, 2025, 2030, 2035, 2040, 2045, 2050]

url = (
    f"https://raw.githubusercontent.com/PyPSA/technology-data/"
    f"master/outputs/costs_{year}.csv"
)

costs = pd.read_csv(url, index_col=[0, 1])

display(costs.head(10))

value           unit  \
technology                        parameter                                    
Alkaline electrolyzer large size  FOM                  6.4000         %/year   
                                  VOM                  0.7799     EUR/MWh_H2   
                                  electricity-input    1.5000  MWh_el/MWh_H2   
                                  investment         903.9648         EUR/kW   
                                  lifetime            40.0000          years   
Alkaline electrolyzer medium size FOM                 18.1000         %/year   
                                  VOM                  0.3033     EUR/MWh_H2   
                                  electricity-input    1.6330  MWh_el/MWh_H2   
                                  investment         718.8102         EUR/kW   
                                  lifetime            20.0000          years   

                                                                                                source  \
technology                        parameter                                                              
Alkaline electrolyzer large size  FOM                JRC, 01_JRC-EU-TIMES Full model, https://zenod...   
                                  VOM                JRC, 01_JRC-EU-TIMES Full model, https://zenod...   
                                  electricity-input  JRC, 01_JRC-EU-TIMES Full model, https://zenod...   
                                  investment         JRC, 01_JRC-EU-TIMES Full model, https://zenod...   
                                  lifetime           JRC, 01_JRC-EU-TIMES Full model, https://zenod...   
Alkaline electrolyzer medium size FOM                JRC, 01_JRC-EU-TIMES Full model, https://zenod...   
                                  VOM                JRC, 01_JRC-EU-TIMES Full model, https://zenod...   
                                  electricity-input  JRC, 01_JRC-EU-TIMES Full model, https://zenod...   
                                  investment         JRC, 01_JRC-EU-TIMES Full model, https://zenod...   
                                  lifetime           JRC, 01_JRC-EU-TIMES Full model, https://zenod...   

                                                          further description  \
technology                        parameter                                     
Alkaline electrolyzer large size  FOM                Reference capacity 72 MW   
                                  VOM                Reference capacity 72 MW   
                                  electricity-input  Reference capacity 72 MW   
                                  investment         Reference capacity 72 MW   
                                  lifetime           Reference capacity 72 MW   
Alkaline electrolyzer medium size FOM                Reference capacity 33 MW   
                                  VOM                Reference capacity 33 MW   
                                  electricity-input  Reference capacity 33 MW   
                                  investment         Reference capacity 33 MW   
                                  lifetime           Reference capacity 33 MW   

                                                     currency_year  
technology                        parameter                         
Alkaline electrolyzer large size  FOM                       2010.0  
                                  VOM                       2025.0  
                                  electricity-input            NaN  
                                  investment                2025.0  
                                  lifetime                     NaN  
Alkaline electrolyzer medium size FOM                       2010.0  
                                  VOM                       2025.0  
                                  electricity-input            NaN  
                                  investment                2025.0  
                                  lifetime                     NaN

In [6]:
# Convert costs expressed per kW to per MW (as per PyPSA's default)
mask = costs["unit"].str.contains("/kW", na=False)
costs.loc[mask, "value"] *= 1e3
costs["unit"] = costs["unit"].str.replace("/kW", "/MW", regex=False)

In [12]:
costs.loc['solar']

,value,unit,source,further description,currency_year
parameter,,,,,
FOM,1.578000e+00,%/year,Calculated. See 'further description'.,Mixed investment costs based on average of 50%...,2020.0
VOM,1.340000e-02,EUR/MWhel,RES costs made up to fix curtailment order,from old pypsa cost assumptions,2025.0
investment,1.018204e+06,EUR/MW_e,Calculated. See 'further description'.,Mixed investment costs based on average of 50%...,2025.0
lifetime,3.500000e+01,years,Calculated. See 'further description'.,Mixed investment costs based on average of 50%...,2020.0


### 2.1 Note that some technology parameters may be spread across different items
For instance, the costs of a CCGT are mostly under the 'CCGT' category, but the cost of the fuel is under 'gas'

In [7]:
costs.loc['CCGT']

,value,unit,source,further description,currency_year
parameter,,,,,
FOM,3.329500e+00,%/year,"Danish Energy Agency, inputs/technology_data_f...","05 Gas turb. CC, steam extract.: Fixed O&M",2015.0
VOM,5.877500e+00,EUR/MWh,"Danish Energy Agency, inputs/technology_data_f...","05 Gas turb. CC, steam extract.: Variable O&M",2025.0
c_b,1.800000e+00,50oC/100oC,"Danish Energy Agency, inputs/technology_data_f...","05 Gas turb. CC, steam extract.: Cb coefficient",2015.0
c_v,1.500000e-01,50oC/100oC,"Danish Energy Agency, inputs/technology_data_f...","05 Gas turb. CC, steam extract.: Cv coefficient",2015.0
efficiency,5.600000e-01,per unit,"Danish Energy Agency, inputs/technology_data_f...","05 Gas turb. CC, steam extract.: Electricity ...",2015.0
investment,1.175507e+06,EUR/MW,"Danish Energy Agency, inputs/technology_data_f...","05 Gas turb. CC, steam extract.: Nominal inve...",2025.0
lifetime,2.500000e+01,years,"Danish Energy Agency, inputs/technology_data_f...","05 Gas turb. CC, steam extract.: Technical li...",2015.0


In [8]:
costs.loc['gas']

,value,unit,source,further description,currency_year
parameter,,,,,
CO2 intensity,0.1980,tCO2/MWh_th,Stoichiometric calculation with 50 GJ/t CH4,NaN,NaN
fuel,11.6278,EUR/MWh_th,"World Bank (2025) Commodity Markets, https://w...","average monthly prices, 1 MWh = 3.412142 MMBTU...",2025.0


Another peculiar case is battery storage, for which the costs of the storage itself and the costs of the inverter are treated separately. This enables representing different Li-ion battery types with different storage durations (e.g., 3h, 6h) by simply multiplying the cost of the storage component by the number of hours, while leaving the cost of the inverter unchanged.

In [10]:
costs.loc['battery storage']

,value,unit,source,further description,currency_year
parameter,,,,,
investment,310195.5,EUR/MWh,"Danish Energy Agency, inputs/technology_data_c...",: Energy storage expansion cost investment,2025.0
lifetime,20.0,years,"Danish Energy Agency, inputs/technology_data_c...",: Technical lifetime,2020.0


In [11]:
costs.loc['battery inverter']

,value,unit,source,further description,currency_year
parameter,,,,,
FOM,0.20,%/year,"Danish Energy Agency, inputs/technology_data_c...",: Fixed O&M,2020.0
efficiency,0.95,per unit,"Danish Energy Agency, inputs/technology_data_c...",: Round trip efficiency DC,2020.0
investment,361003.40,EUR/MW,"Danish Energy Agency, inputs/technology_data_c...",: Output capacity expansion cost investment,2025.0
lifetime,10.00,years,"Danish Energy Agency, inputs/technology_data_c...",: Technical lifetime,2020.0


## 3. Packing everything into a simple Excel table to reuse in our model
Now, let's extract what we may need for a (simplified) national model of the Netherlands and export it for later use

In [15]:
techs_of_interest = ['ccgt', 'coal', 'biomass', 'solar', 'wind-onshore', 'wind-offshore' ,'nuclear', 'battery-4h']
params_of_interest = ['fuel', 'VOM', 'efficiency', 'FOM', 'CO2 intensity', 'lifetime', 'investment', 'installed_capacity']

technoecon_params = pd.DataFrame(index=techs_of_interest, columns=params_of_interest).astype(float).fillna(0)
technoecon_params

,fuel,VOM,efficiency,FOM,CO2 intensity,lifetime,investment,installed_capacity
ccgt,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
solar,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
wind-onshore,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
wind-offshore,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
nuclear,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
battery,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [31]:
for param in ['VOM', 'FOM', 'efficiency', 'lifetime', 'investment']:
    technoecon_params.loc['ccgt',param] = costs.loc['CCGT'].loc[param]['value']
    technoecon_params.loc['biomass',param] = costs.loc['biomass CHP'].loc[param]['value']
    technoecon_params.loc['coal',param] = costs.loc['coal'].loc[param]['value']
    technoecon_params.loc['nuclear',param] = costs.loc['nuclear'].loc[param]['value']
    if param == 'lifetime':
        technoecon_params.loc['battery',param] = (
            costs.loc['battery inverter'].loc[param]['value'] + 
            costs.loc['battery storage'].loc[param]['value']
        )/2 # average of the two lifetimes
    elif param == 'investment':
        technoecon_params.loc['battery',param] = (
            costs.loc['battery inverter'].loc[param]['value'] + 
            costs.loc['battery storage'].loc[param]['value']*4 #number of hours of storage in this example
        )
    else:
        try:
            technoecon_params.loc['battery',param] = costs.loc['battery inverter'].loc[param]['value']
        except:
            continue
    
for param in ['VOM', 'FOM', 'lifetime', 'investment']:
    technoecon_params.loc['solar',param] = costs.loc['solar'].loc[param]['value']
    technoecon_params.loc['wind-onshore',param] = costs.loc['onwind'].loc[param]['value']
    technoecon_params.loc['wind-offshore',param] = costs.loc['offwind'].loc[param]['value']

for param in ['CO2 intensity', 'fuel']:
    technoecon_params.loc['ccgt',param] = costs.loc['gas'].loc[param]['value']
    technoecon_params.loc['coal',param] = costs.loc['coal'].loc[param]['value']
for param in ['fuel']:
    technoecon_params.loc['nuclear',param] = costs.loc['nuclear'].loc[param]['value']
    technoecon_params.loc['biomass',param] = costs.loc['biomass'].loc[param]['value']

In [32]:
technoecon_params

,fuel,VOM,efficiency,FOM,CO2 intensity,lifetime,investment,installed_capacity
ccgt,11.6278,5.8775,0.5600,3.3295,0.1980,25.0,1175506.7,24354.5
solar,0.0000,0.0134,0.0000,1.5780,0.0000,35.0,1018204.4,7366.1
wind-onshore,0.0000,2.0037,0.0000,1.2514,0.0000,27.0,1494463.1,5505.8
wind-offshore,0.0000,0.0267,0.0000,2.5093,0.0000,27.0,2505378.1,7138.0
nuclear,7.4536,4.4590,0.3260,1.2700,0.0000,40.0,10805703.8,515.0
battery,0.0000,0.0000,0.9500,0.2000,0.0000,15.0,1601785.4,263.4
biomass,9.3506,2.8138,0.2994,3.6081,NaN,25.0,4516713.3,NaN
coal,7.8202,4.1005,0.3560,1.3100,0.3361,40.0,4812024.4,NaN


In [33]:
for param in ['installed_capacity']:
    technoecon_params.loc['nuclear',param] = capacity_by_fuel['Nuclear']
    # For simplicity, in this example, we add the small Oil capacity to the CCGT
    technoecon_params.loc['ccgt',param] = capacity_by_fuel[['Natural Gas', 'Oil']].sum()
    # Similarly, we add together various types of biomass and waste
    technoecon_params.loc['biomass',param] = capacity_by_fuel[['Solid Biomass', 'Biogas', 'Waste']].sum()
    technoecon_params.loc['coal',param] = capacity_by_fuel['Hard Coal']
    technoecon_params.loc['solar',param] = capacity_by_fuel['Solar']
    technoecon_params.loc['wind-onshore',param] = capacity_by_technology['Onshore']
    technoecon_params.loc['wind-offshore',param] = capacity_by_technology['Offshore']
    technoecon_params.loc['battery',param] = capacity_by_technology['Li']

In [34]:
technoecon_params

,fuel,VOM,efficiency,FOM,CO2 intensity,lifetime,investment,installed_capacity
ccgt,11.6278,5.8775,0.5600,3.3295,0.1980,25.0,1175506.7,17470.5
solar,0.0000,0.0134,0.0000,1.5780,0.0000,35.0,1018204.4,7366.1
wind-onshore,0.0000,2.0037,0.0000,1.2514,0.0000,27.0,1494463.1,5505.8
wind-offshore,0.0000,0.0267,0.0000,2.5093,0.0000,27.0,2505378.1,7138.0
nuclear,7.4536,4.4590,0.3260,1.2700,0.0000,40.0,10805703.8,515.0
battery,0.0000,0.0000,0.9500,0.2000,0.0000,15.0,1601785.4,263.4
biomass,9.3506,2.8138,0.2994,3.6081,NaN,25.0,4516713.3,1573.2
coal,7.8202,4.1005,0.3560,1.3100,0.3361,40.0,4812024.4,6007.0


In [35]:
technoecon_params.to_csv('input_data/technoecon_params.csv')